# Convolutional Autoencoder for Image Denoising
## Neural networks for energy applications — Day 4, Exercise 3

In this exercise we build a **convolutional autoencoder** that learns to denoise
images. We train the network to reconstruct clean MNIST digits from noisy inputs,
using Conv2d layers in the encoder and ConvTranspose2d layers in the decoder.
We then explore the latent space to understand what the model has learned.

**Evaluation protocol.** We hold out 5,000 of the 60,000 training images as a
**validation set**. Validation images are used to monitor training, to keep the best
epoch and to compare latent sizes, always with the same fixed noise. The MNIST
**test set** is used once, in Section 8, for the model we have already chosen.

**Runtime.** Budgets are set for a laptop CPU: 8 epochs for the main model and
3 epochs for each model in the latent-size comparison.
The notebook runs in about 5–8 minutes on a laptop CPU and in about 1 minute on a GPU
or Apple silicon. With more time, raise `NUM_EPOCHS` and `COMPARE_EPOCHS`.

---
## 0 — Imports and Setup

In [ ]:
import copy
import os

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split

import torchvision
import torchvision.transforms as transforms

from sklearn.manifold import TSNE

# Plotting defaults
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 12,
})

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# Use a GPU if there is one (CUDA, or MPS on Apple silicon), otherwise the CPU.
# GPU results can differ slightly from CPU results even with the same seed.
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# MNIST is downloaded to ./data unless the COURSE_DATA environment variable names another folder
DATA_ROOT = os.environ.get("COURSE_DATA", "./data")
print(f"Data folder: {DATA_ROOT}")
print("Setup complete.")

---
## 1 — Load MNIST Dataset

In [ ]:
# YOUR CODE HERE
# TODO: Define a simple transform with transforms.ToTensor() (pixels in [0, 1])
# TODO: Load the MNIST train and test sets with torchvision.datasets.MNIST
#       using root=DATA_ROOT, download=True
# TODO: Hold out 5,000 training images for validation:
#       train_dataset, val_dataset = random_split(full_train_dataset, [55000, 5000],
#                                                 generator=torch.Generator().manual_seed(RANDOM_STATE))
# TODO: Create train_loader with BATCH_SIZE=128, shuffle=True, num_workers=0
#       (the test set stays untouched until Section 8)
# TODO: Print the number of training, validation and test samples and the image shape

---
## 2 — Add Noise to Images

We add Gaussian noise to the input images. The autoencoder will learn to map
noisy images back to clean originals.

Training batches get **fresh noise every time** they are drawn. The validation images
get **one fixed noise draw** (seeded), so that every epoch and every model in Section 6
is scored on exactly the same noisy inputs; otherwise part of a difference between two
models would just be different noise.

In [ ]:
# YOUR CODE HERE
NOISE_FACTOR = 0.5
# TODO: Write add_noise(images, noise_factor=NOISE_FACTOR, generator=None) that adds Gaussian
#       noise and clamps the result to [0, 1]
#       Hint: without a generator use torch.randn_like(images) (fresh noise for training batches);
#             with a seeded generator use torch.randn(images.shape, generator=generator)
#       Hint: use torch.clamp(noisy, 0.0, 1.0)
# TODO: Build FIXED noisy validation images so every model is scored on the same inputs:
#       val_clean = torch.stack([image for image, _ in val_dataset])
#       val_labels = torch.tensor([label for _, label in val_dataset])
#       val_noisy = add_noise(val_clean, generator=torch.Generator().manual_seed(RANDOM_STATE))
#       val_loader = DataLoader(TensorDataset(val_noisy, val_clean), batch_size=BATCH_SIZE, shuffle=False)
# TODO: Visualize 8 clean vs 8 noisy validation images side by side in a 2x8 grid
# TODO: Print the MSE of the noisy inputs against the clean images (the "do nothing" reference)

---
## 3 — Build Convolutional Autoencoder

The **encoder** compresses the 28x28 image down to a latent representation using
Conv2d layers. The **decoder** reconstructs the image using ConvTranspose2d layers.

Architecture:

| Component | Layer | Output Shape |
|-----------|-------|--------------|
| Encoder | Conv2d(1, 16, 3, stride=2, pad=1) + ReLU | 16 x 14 x 14 |
| Encoder | Conv2d(16, 32, 3, stride=2, pad=1) + ReLU | 32 x 7 x 7 |
| Encoder | Conv2d(32, 64, 7) + ReLU | 64 x 1 x 1 |
| Decoder | ConvTranspose2d(64, 32, 7) + ReLU | 32 x 7 x 7 |
| Decoder | ConvTranspose2d(32, 16, 3, stride=2, pad=1, out_pad=1) + ReLU | 16 x 14 x 14 |
| Decoder | ConvTranspose2d(16, 1, 3, stride=2, pad=1, out_pad=1) + Sigmoid | 1 x 28 x 28 |

In [ ]:
# YOUR CODE HERE
# TODO: Define class ConvAutoencoder(nn.Module) with __init__(self, latent_dim=64):
#   Encoder (nn.Sequential):
#     - Conv2d(1, 16, kernel_size=3, stride=2, padding=1) + ReLU
#     - Conv2d(16, 32, kernel_size=3, stride=2, padding=1) + ReLU
#     - Conv2d(32, latent_dim, kernel_size=7) + ReLU
#   Decoder (nn.Sequential):
#     - ConvTranspose2d(latent_dim, 32, kernel_size=7) + ReLU
#     - ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1) + ReLU
#     - ConvTranspose2d(16, 1, 3, stride=2, padding=1, output_padding=1) + Sigmoid
#   Methods: encode(x), decode(z), forward(x)
# TODO: Instantiate with latent_dim=64, send to device, print model and param count
# TODO: Verify the input, latent (model.encode) and output shapes with a dummy input of shape (1, 1, 28, 28)


---
## 4 — Train the Denoising Autoencoder

We train with **MSE loss** between the reconstructed output and the clean original
image. The input to the model is the noisy version.

After every epoch we compute the MSE on the fixed noisy **validation** images and keep a
copy of the weights from the epoch with the lowest validation MSE. The test set plays
no part in this choice.

### 4.1 Training Loop

In [ ]:
# YOUR CODE HERE
# TODO: Define criterion = nn.MSELoss() and optimizer = Adam(lr=1e-3)
# TODO: Write denoising_mse(model, loader, criterion) that returns the mean MSE between
#       model(noisy) and clean over a loader of (noisy, clean) pairs
#       Hint: model.eval() and torch.no_grad(); weight each batch loss by its batch size
# TODO: Train for NUM_EPOCHS=8 epochs (classroom budget; more epochs usually give sharper reconstructions):
#   For each batch: add FRESH noise to images, pass noisy through model,
#   compute MSE between output and CLEAN images, backprop
# TODO: After each epoch compute the validation loss with denoising_mse(model, val_loader, criterion)
#       (the test set is not used here)
# TODO: Store train_loss and val_loss in a history dict
# TODO: Keep a copy of the best weights: best_state = copy.deepcopy(model.state_dict())
#       whenever the validation loss improves; reload them with model.load_state_dict(best_state)
# Hint: loss = criterion(model(noisy_images), clean_images)

### 4.2 Plot Training Curve

In [ ]:
# YOUR CODE HERE
# TODO: Plot train and validation MSE loss vs epoch
# TODO: Mark the kept epoch (best_epoch) with a vertical line

---
## 5 — Visualize Denoising Results

We compare noisy input, reconstructed output, clean original and the absolute
**residual** |reconstruction − clean| for eight validation digits. Structure in the
residual (strokes, edges) means the model misses signal; noise-like residuals mean it
does not.

In [ ]:
# YOUR CODE HERE
# TODO: Reconstruct the first 8 fixed noisy validation images (val_noisy[:8]) with the model
# TODO: Compute the residual = |reconstructed - clean|
# TODO: Display the 8 images in a 4-row grid:
#       Row 1: noisy input
#       Row 2: reconstructed output
#       Row 3: clean original
#       Row 4: residual
# Hint: use model.eval() and torch.no_grad() for inference; imshow(..., vmin=0, vmax=1)

---
## 6 — Experiment with Latent Space Dimension

We train autoencoders with latent sizes 8, 32 and 64 to see how the compression level
affects reconstruction quality. For a fair comparison every size gets the **same
training budget** (3 epochs, the same seed) and is scored on the **same fixed noisy
validation images**.

In [ ]:
# YOUR CODE HERE
# TODO: For latent_dims = [8, 32, 64], with the same budget COMPARE_EPOCHS = 3:
#   - Call torch.manual_seed(RANDOM_STATE), then create a new ConvAutoencoder(latent_dim=ldim)
#   - Train it on the denoising task for COMPARE_EPOCHS epochs
#   - Evaluate the validation MSE with denoising_mse(ae, val_loader, criterion)
# TODO: Print a comparison table of latent_dim, parameter count and validation MSE
# Hint: store each trained model in a results dict for visualization

In [ ]:
# YOUR CODE HERE
# TODO: Visualize reconstructions of the same 8 validation images for each latent dimension:
#   Row 0: noisy input (val_noisy[:8])
#   Rows 1-3: reconstructed images for each latent_dim
#   Last row: clean original (val_clean[:8])

---
## 7 — Visualize Latent Representations

We extract the latent vectors of the validation images (encoded from the clean images)
and use t-SNE to project them into 2D, colouring by digit class.

t-SNE is a visualization aid only: the distances between clusters and their sizes are
not meaningful, and a tidy map is not evidence that the codes are useful for another task.

In [ ]:
# YOUR CODE HERE
# TODO: Extract latent vectors of the validation images (val_clean) from the main model
#       (latent_dim=64) using model.encode(); loop over a DataLoader of (val_clean, val_labels)
# TODO: Flatten the latent output from (batch, 64, 1, 1) to (batch, 64)
# TODO: Concatenate all batches into a single numpy array
# Hint: use model.eval() and torch.no_grad()

In [ ]:
# YOUR CODE HERE
# TODO: Take a random subset of 2000 validation samples for t-SNE
#       Hint: rng = np.random.default_rng(RANDOM_STATE); rng.choice(..., replace=False)
# TODO: Run TSNE(n_components=2, random_state=RANDOM_STATE) on the latent subset
# TODO: Create a scatter plot coloured by digit class
# Hint: use cmap="tab10" and add a colorbar

---
## 8 — Final Check on the Test Set (once)

All choices are now locked: the latent-64 model from Section 4 with the weights of its
best validation epoch. (Had the comparison in Section 6 favoured another size, we would
retrain that size with the full budget and check it on validation data before this step.)

We now use the MNIST test set for the first and only time, with its own fixed noise, and
report the test MSE next to the "do nothing" reference: the MSE of the noisy input itself.

In [ ]:
# YOUR CODE HERE
# TODO: Build fixed noisy TEST images (use a different seed, e.g. RANDOM_STATE + 1):
#       test_clean = torch.stack([image for image, _ in test_dataset])
#       test_noisy = add_noise(test_clean, generator=torch.Generator().manual_seed(RANDOM_STATE + 1))
# TODO: Create test_loader from TensorDataset(test_noisy, test_clean)
# TODO: Print the MSE of the noisy input itself (do-nothing reference) and the final model's
#       test MSE from denoising_mse(model, test_loader, criterion). Do this once, at the end.

---
## Summary

In this exercise we:

1. **Loaded MNIST**, held out 5,000 training images for validation and added Gaussian noise.
2. **Built a convolutional autoencoder** with Conv2d encoder and ConvTranspose2d decoder.
3. **Trained** the model to reconstruct clean images from noisy inputs using MSE loss,
   monitoring fixed noisy validation images and keeping the best validation epoch.
4. **Visualized** denoising results: noisy input, reconstruction, clean original and residual.
5. **Compared** latent sizes 8, 32 and 64 with the same training budget on the same
   validation images.
6. **Visualized** the latent space of validation images using t-SNE.
7. **Checked the locked model once on the test set**, against the do-nothing reference.

Key takeaways:
- Convolutional autoencoders preserve spatial structure better than fully connected ones.
- The latent dimension controls the information bottleneck: too small loses detail,
  too large may not denoise as effectively.
- Compare models on the same validation images with the same noise and the same
  training budget; the test set is for one final check.
- Similar digits tend to lie close together in the t-SNE map, but t-SNE distances and
  cluster sizes are not evidence about the latent space on their own.